# Explore

Scratch space. Anything that turns out to be worth keeping graduates to a
script under `scripts/`; this notebook is not the place where the pipeline
eventually lives.

In [1]:
from pathlib import Path

import pandas as pd

# The notebook runs from notebooks/, so paths are resolved against the repo
# root instead of the working directory.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RACE_RESULTS = REPO_ROOT / "data" / "raw" / "race_results"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df = pd.read_parquet(RACE_RESULTS)
df.shape

(3700, 39)

In [2]:
df.head()

,number,position,positionText,points,grid,laps,status,driverId,driverNumber,driverCode,driverUrl,givenName,familyName,dateOfBirth,driverNationality,constructorId,constructorUrl,constructorName,constructorNationality,totalRaceTimeMillis,totalRaceTime,fastestLapRank,fastestLapNumber,fastestLapTime,fastestLapAvgSpeedUnits,fastestLapAvgSpeed,season,round,raceUrl,raceName,raceDate,raceTime,circuitId,circuitUrl,circuitName,lat,long,locality,country
0,5,1,1,25.0,3,58,Finished,vettel,5,VET,http://en.wikipedia.org/wiki/Sebastian_Vettel,Sebastian,Vettel,1987-07-03,German,ferrari,https://en.wikipedia.org/wiki/Scuderia_Ferrari,Ferrari,Italian,5373283.0,0 days 01:29:33.283000,4.0,53.0,0 days 00:01:26.469000,kph,220.782,2018,1,https://en.wikipedia.org/wiki/2018_Australian_...,Australian Grand Prix,2018-03-25,05:10:00,albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,Albert Park Grand Prix Circuit,-37.8497,144.968,Melbourne,Australia
1,44,2,2,18.0,1,58,Finished,hamilton,44,HAM,http://en.wikipedia.org/wiki/Lewis_Hamilton,Lewis,Hamilton,1985-01-07,British,mercedes,https://en.wikipedia.org/wiki/Mercedes-Benz_in...,Mercedes,German,5378319.0,0 days 00:00:05.036000,3.0,50.0,0 days 00:01:26.444000,kph,220.845,2018,1,https://en.wikipedia.org/wiki/2018_Australian_...,Australian Grand Prix,2018-03-25,05:10:00,albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,Albert Park Grand Prix Circuit,-37.8497,144.968,Melbourne,Australia
2,7,3,3,15.0,2,58,Finished,raikkonen,7,RAI,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C...,Kimi,Räikkönen,1979-10-17,Finnish,ferrari,https://en.wikipedia.org/wiki/Scuderia_Ferrari,Ferrari,Italian,5379592.0,0 days 00:00:06.309000,2.0,57.0,0 days 00:01:26.373000,kph,221.027,2018,1,https://en.wikipedia.org/wiki/2018_Australian_...,Australian Grand Prix,2018-03-25,05:10:00,albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,Albert Park Grand Prix Circuit,-37.8497,144.968,Melbourne,Australia
3,3,4,4,12.0,8,58,Finished,ricciardo,3,RIC,http://en.wikipedia.org/wiki/Daniel_Ricciardo,Daniel,Ricciardo,1989-07-01,Australian,red_bull,https://en.wikipedia.org/wiki/Red_Bull_Racing,Red Bull,Austrian,5380352.0,0 days 00:00:07.069000,1.0,54.0,0 days 00:01:25.945000,kph,222.128,2018,1,https://en.wikipedia.org/wiki/2018_Australian_...,Australian Grand Prix,2018-03-25,05:10:00,albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,Albert Park Grand Prix Circuit,-37.8497,144.968,Melbourne,Australia
4,14,5,5,10.0,10,58,Finished,alonso,14,ALO,http://en.wikipedia.org/wiki/Fernando_Alonso,Fernando,Alonso,1981-07-29,Spanish,mclaren,https://en.wikipedia.org/wiki/McLaren,McLaren,British,5401169.0,0 days 00:00:27.886000,7.0,57.0,0 days 00:01:26.978000,kph,219.489,2018,1,https://en.wikipedia.org/wiki/2018_Australian_...,Australian Grand Prix,2018-03-25,05:10:00,albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,Albert Park Grand Prix Circuit,-37.8497,144.968,Melbourne,Australia


In [3]:
df.columns.tolist()

['number',
 'position',
 'positionText',
 'points',
 'grid',
 'laps',
 'status',
 'driverId',
 'driverNumber',
 'driverCode',
 'driverUrl',
 'givenName',
 'familyName',
 'dateOfBirth',
 'driverNationality',
 'constructorId',
 'constructorUrl',
 'constructorName',
 'constructorNationality',
 'totalRaceTimeMillis',
 'totalRaceTime',
 'fastestLapRank',
 'fastestLapNumber',
 'fastestLapTime',
 'fastestLapAvgSpeedUnits',
 'fastestLapAvgSpeed',
 'season',
 'round',
 'raceUrl',
 'raceName',
 'raceDate',
 'raceTime',
 'circuitId',
 'circuitUrl',
 'circuitName',
 'lat',
 'long',
 'locality',
 'country']

In [4]:
df[df.driverId == 'leclerc'].sort_values(['season', 'round'])[['season', 'round', 'grid', 'position']].head(15)

,season,round,grid,position
12,2018,1,18,13
31,2018,2,19,12
58,2018,3,19,19
65,2018,4,13,6
89,2018,5,14,10
117,2018,6,14,18
129,2018,7,13,10
149,2018,8,8,10
168,2018,9,18,9
198,2018,10,9,19


## Label

No non-finisher is ever classified inside the top 10 in this data, so
`position <= 10` needs no separate handling for retirements.

In [5]:
df['top10'] = (df.position <= 10).astype(int)
df['top10'].mean()

np.float64(0.4972972972972973)

## First lagged feature

Share of the driver's own previous 5 races that ended in the top 10.

Both `shift` and `rolling` run inside the groupby. Chaining `.rolling()` onto
the result of `groupby(...).shift(1)` instead would run the window over the
whole frame, blind to where one driver ends and the next begins. That happens
to give the right numbers while the NaN that `shift` leaves at each driver's
first race sits inside the window, and stops doing so the moment anything
shortens the window, `min_periods=1` above all.

**Open: this window crosses seasons.** A driver's first race of 2019 is scored
from their last five of 2018, which may be a different team and a different
car. Not yet decided.

In [6]:
df = df.sort_values(['driverId', 'season', 'round'])

df['top10_rate_last5'] = df.groupby('driverId')['top10'].transform(
    lambda s: s.shift(1).rolling(5).mean()
)

In [7]:
df[df.driverId == 'leclerc'].sort_values(['season', 'round'])[
    ['season', 'round', 'position', 'top10', 'top10_rate_last5']
].head(8)

,season,round,position,top10,top10_rate_last5
12,2018,1,13,0,NaN
31,2018,2,12,0,NaN
58,2018,3,19,0,NaN
65,2018,4,6,1,NaN
89,2018,5,10,1,NaN
117,2018,6,18,0,0.4
129,2018,7,10,1,0.4
149,2018,8,10,1,0.6


## Does the current race leak in?

Counting NaNs shows the window has warmed up, but says nothing about whether
the numbers that are not NaN are the right numbers. The second cell answers
that directly, by rebuilding the feature from scratch in plain Python and
comparing.

`aitken` and `pietro_fittipaldi` report fewer than 5 because they started
fewer than 5 races in total, so `head(5)` has nothing to return.

In [8]:
df.groupby('driverId')['top10_rate_last5'].apply(lambda x: x.head(5).isna().sum())

driverId
aitken               1
albon                5
alonso               5
antonelli            5
arvid_lindblad       5
bearman              5
bortoleto            5
bottas               5
brendon_hartley      5
colapinto            5
de_vries             5
doohan               5
ericsson             5
gasly                5
giovinazzi           5
grosjean             5
hadjar               5
hamilton             5
hulkenberg           5
kevin_magnussen      5
kubica               5
kvyat                5
latifi               5
lawson               5
leclerc              5
max_verstappen       5
mazepin              5
mick_schumacher      5
norris               5
ocon                 5
perez                5
piastri              5
pietro_fittipaldi    2
raikkonen            5
ricciardo            5
russell              5
sainz                5
sargeant             5
sirotkin             5
stroll               5
tsunoda              5
vandoorne            5
vettel               5
zh

In [9]:
expected = []
for _, g in df.groupby('driverId', sort=False):
    t = g['top10'].tolist()
    for i in range(len(t)):
        expected.append(sum(t[i - 5:i]) / 5 if i >= 5 else float('nan'))

df['top10_rate_last5'].round(10).equals(pd.Series(expected, index=df.index).round(10))

True